In [ ]:
import pandas as pd
import time
import torch
from darts import TimeSeries
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

import pandas as pd
df = pd.read_csv("data_train/25116_1_3_3/mmwave_ss.csv")
df["kss_score"] = 4
df["log_time"] = pd.to_datetime(df['log_time'])
df = df.set_index("log_time")
df = df.resample("1s").mean().interpolate(method="linear")
df = df.reset_index()

target_ts = TimeSeries.from_dataframe(df, time_col="log_time", value_cols=["kss_score"])
past_cov_ts = TimeSeries.from_dataframe(df, time_col="log_time", value_cols=["breath_rate", "heart_rate"])

scaler_target = Scaler()
scaler_past_cov = Scaler()

target_scaled = scaler_target.fit_transform(target_ts)
past_cov_scaled = scaler_past_cov.fit_transform(past_cov_ts)

In [ ]:
import numpy as np

min_per_col = np.min(past_cov_ts.values(), axis=0)
max_per_col = np.max(past_cov_ts.values(), axis=0)

print(f"Min per kolom: {min_per_col}")
print(f"Max per kolom: {max_per_col}")

In [ ]:
timestamp = time.time()
logger = CSVLogger(save_dir="logs/", name=f"tft_run_{timestamp}")

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1, # only save 1 file
    dirpath="logs/checkpoints/",
    filename=f"tft_{timestamp}_epoch{{epoch:02d}}_val{{val_loss:.4f}}",
    save_weights_only=False 
)

model = TFTModel(
    input_chunk_length=120, # change to seq_len * context_time
    output_chunk_length=30, # change to seq_len * prediction_time
    hidden_size=32,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=64, # change to batch_size
    n_epochs=5, # change to epoch
    optimizer_kwargs={"lr": 1e-3},
    loss_fn=torch.nn.MSELoss(),
    add_encoders={ # automate extract future_cov from timestamp/log_time
        'cyclic': {'future': ['minute', 'second', 'hour']},
        'transformer': Scaler()
    },
    
    pl_trainer_kwargs={ # trainer from pytorch lightning
        "accelerator": "auto",
        "callbacks": [checkpoint_callback],
        "logger": logger,
        "enable_checkpointing":True,
        "log_every_n_steps": 1
    },
    random_state=42, # seed so the experiment can be reproduced
)

In [ ]:
model.fit(
    series=target_ts,
    past_covariates=past_cov_ts,
    val_series=target_ts,
    val_past_covariates=past_cov_ts,
    verbose=True,
    max_samples_per_ts=100 # get 1000 random windows for training
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_df = pd.read_csv("logs/tft_run_1779512223.6690488/version_0/metrics.csv")
def visualize_metrics(file):
    metrics_df = pd.read_csv(f"{file}")
    metrics_df = metrics_df.groupby("epoch").agg({
        "train_loss": "mean",
        "val_loss": "first"
    }).reset_index()

    plt.figure(figsize=(10, 6))
    plt.plot(metrics_df['epoch'], metrics_df['train_loss'], 
            label='Train Loss', marker='o', linewidth=2, color='tab:blue')
    plt.plot(metrics_df['epoch'], metrics_df['val_loss'], 
            label='Validation Loss', marker='s', linewidth=2, color='tab:orange')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss Value', fontsize=12)
    plt.title('Training Results: Train Loss vs Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    os.makedirs("results_metrics", exist_ok=True)
    image_name, _ = os.path.splitext(os.path.basename(file))
    plt.savefig(f"results_metrics/{image_name}.jpg", bbox_inches='tight', dpi=300)



In [ ]:
model = "logs/checkpoints/tft_1779512223.6690488_epochepoch=03_valval_loss=0.0013.ckpt"
timestamp = model.split("_")[1]
print(timestamp)